# Tracing Data Flow in Hyrax

When working with a new dataset or model it can be hard to know whether your data is being processed correctly.
Hyrax provides a **trace mode** that lets you follow a small batch of data items through the entire pipeline —
from raw dataset access, through batching and input preparation, all the way to model evaluation — so you can
inspect what is actually happening at each step.

This notebook walks through a typical trace session:

1. Setting up a Hyrax instance
2. Running a verb with `trace=N` to capture pipeline data
3. Printing and navigating the returned `TraceResult`
4. Drilling into individual stages and function calls

Trace mode is intended for interactive use in notebooks — it is not a production profiling or logging tool.

## Install Hyrax

Skip this step if you have already installed Hyrax in your environment.

In [ ]:
# %pip install hyrax

## Create a Hyrax instance

We start by creating a `Hyrax` instance and pointing it at a simple built-in model and dataset.
Here we use `HyraxAutoencoder` trained on `HyraxRandomDataset` — a dataset that generates random
tensors without requiring any downloaded data, which makes it convenient for experimentation.

You can replace this with your own model and dataset configuration to trace your real workflow.

In [ ]:
import hyrax

h = hyrax.Hyrax()

## Configure the model and data

We will use the `HyraxAutoencoder` model and `HyraxRandomDataset` to keep this notebook self-contained.
The random dataset produces small random tensors (`[1, 32, 32]`) that resemble single-band image cutouts.

In [ ]:
h.config["model"]["name"] = "HyraxAutoencoder"

# Use a small 1-channel 32×32 image shape to keep things fast
h.config["data_set"]["HyraxRandomDataset"]["shape"] = [1, 32, 32]
h.config["data_set"]["HyraxRandomDataset"]["size"] = 50
h.config["data_set"]["HyraxRandomDataset"]["seed"] = 42

data_request = {
    "train": {
        "data": {
            "dataset_class": "HyraxRandomDataset",
            "data_location": "./trace_data",
            "fields": ["image"],
            "primary_id_field": "object_id",
            "split_fraction": 1.0,
        },
    },
    "infer": {
        "data": {
            "dataset_class": "HyraxRandomDataset",
            "data_location": "./trace_data",
            "fields": ["image"],
            "primary_id_field": "object_id",
            "split_fraction": 1.0,
        },
    },
}
h.set_config("data_request", data_request)

## Running a verb in trace mode

Any instrumented verb (`train`, `infer`, `test`) accepts a `trace=N` keyword argument.
`N` controls how many data items are traced through the pipeline — keep this small (2–10)
to get a readable output.

When `trace=N` is passed, the verb:
- Processes only a single batch of `N` items instead of the full dataset
- Returns a `TraceResult` object instead of the usual verb return value

The `TraceResult` captures the inputs and outputs of every major pipeline step.

In [ ]:
trace_result = h.train(trace=2)

## Printing the trace

Printing `trace_result` gives a high-level summary of all pipeline stages and the function calls
captured within them.  Each entry shows the function name, its input and output names, the
shapes / hashes of any tensors, and how long the call took.

In [ ]:
print(trace_result)

The output lists five stages in pipeline order:

| Stage | What is captured |
|---|---|
| `dataset_getter` | Individual `get_*` calls on the dataset class |
| `resolve_data` | `DataProvider.resolve_data` — assembles all fields for each item |
| `collate` | `DataProvider.collate` and `handle_nans` — builds batch tensors |
| `prepare_inputs` | Model's `prepare_inputs` — converts the data dictionary to model input tensors |
| `evaluation` | Model functions such as `forward`, `train_batch`, etc. |

The table above is printed as text from the `TraceResult.__str__` implementation so that the
notebook cell output and a plain `print()` call look identical.

## Exploring stages

You can access any stage using either attribute access or dictionary-style access — both are equivalent.
Tab completion in a notebook environment will suggest the valid stage names.

In [ ]:
# Attribute-style access
collate_stage = trace_result.collate
print(collate_stage)

In [ ]:
# Dictionary-style access — identical result
evaluation_stage = trace_result["evaluation"]
print(evaluation_stage)

Each stage is a `TraceStage` — a list of `TraceCall` records in the order they were executed.
You can ask how many calls were captured:

In [ ]:
print(f"resolve_data calls : {len(trace_result.resolve_data)}")
print(f"collate calls      : {len(trace_result.collate)}")
print(f"evaluation calls   : {len(trace_result.evaluation)}")

## Exploring individual function calls

Within a stage you can index calls by number (call order) or by function name.
A `TraceCall` captures the function's argument values and return value along with timing information.

In [ ]:
# Get the first call in the collate stage
first_collate_call = trace_result.collate[0]
print(first_collate_call)

In [ ]:
# Access the captured batch tensor directly — attribute and dict access both work
batch_dict = first_collate_call.batch_dict
print(type(batch_dict))
print(batch_dict)

In [ ]:
# Get a list of all calls to a particular function by providing the display name
# (visible from the print output above)
all_resolve_calls = trace_result.resolve_data["DataProvider__resolve_data"]
print(f"Number of resolve_data calls: {len(all_resolve_calls)}")
print(all_resolve_calls[0])

## Tracing other verbs

The `trace=N` keyword works the same way for `infer` and `test`:

In [ ]:
infer_trace = h.infer(trace=2)
print(infer_trace)

## Instrumenting custom models and datasets

If you have written a custom model or dataset class you can opt specific methods into trace capture
using the `@trace_model_func` and `@trace_dataset_func` decorators.

```python
from hyrax.trace import trace_model_func, trace_dataset_func

class MyModel(nn.Module):
    @trace_model_func
    def my_custom_forward(self, batch):
        # This call will appear in the 'evaluation' stage of the TraceResult
        ...

class MyDataset(HyraxDataset):
    @trace_dataset_func
    def get_image(self, index):
        # This call will appear in the 'dataset_getter' stage of the TraceResult
        ...
```

The decorators add a small overhead to every call, so they are intended for use during development
and debugging rather than in production.  Remove them (or use them selectively) once you are satisfied
with how your data is flowing.